# Forward-Risk-Manager: Intensive Colab T4 Runbook

This notebook runs the full workflow end-to-end with Colab-safe defaults:
1. Environment and GPU checks
2. Drive mount and repo discovery
3. Dependency install + editable package install + tests
4. Build/train/benchmark/sweep (+ optional sweep promotion + retrain)
5. Goodness backtest + constrained scenarios + stress test
6. Hallucination calibration + diagnostics
7. Artifact bundle export and download

Notes:
- Ticker selection defaults to `AUTO` (no hard dependency on `MDY`).
- If Colab Python is `<3.11`, setup installs a temporary `tomllib` compatibility shim.


## 1) Runtime Setup

In Colab:
- Runtime -> Change runtime type -> Hardware accelerator: `GPU`
- Prefer `T4` and high-RAM runtime when available


In [8]:
import os
import subprocess
import sys

# CUDA/runtime friendliness
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "max_split_size_mb:256")

print("python:", sys.version)
print("python_executable:", sys.executable)

if sys.version_info < (3, 10):
    raise RuntimeError("Python >= 3.10 is required.")

if sys.version_info < (3, 11):
    print("INFO: Python < 3.11 detected. A tomllib compatibility shim will be created in setup.")

try:
    import torch
except Exception as exc:
    raise RuntimeError("PyTorch is required in Colab runtime.") from exc

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_version:", torch.version.cuda)

try:
    print(subprocess.check_output(["nvidia-smi", "-L"], text=True))
except Exception as exc:
    print("nvidia-smi unavailable:", exc)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Enable GPU runtime and rerun.")

gpu_name = torch.cuda.get_device_name(0)
print("gpu:", gpu_name)
if "T4" not in gpu_name.upper():
    print("WARNING: GPU is not T4. Notebook still works; timings will differ.")

# Throughput-friendly settings for T4
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True


python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
python_executable: /usr/bin/python3
torch: 2.9.0+cu128
cuda_available: True
cuda_version: 12.8
GPU 0: Tesla T4 (UUID: GPU-9b8fb6b4-f1d1-f657-6e67-ee8f69506422)

gpu: Tesla T4


## 2) Mount Drive And Open Repo


In [9]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

    repo_candidates = [
        "/content/drive/MyDrive/Forward-Risk-Manager",
        "/content/drive/MyDrive/forward-risk-manager",
        "/content/Forward-Risk-Manager",
    ]
    existing = [p for p in repo_candidates if Path(p).exists()]

    if existing:
        REPO_DIR = existing[0]
    else:
        drive_root = Path("/content/drive/MyDrive")
        fuzzy = []
        if drive_root.exists():
            fuzzy = sorted(str(p) for p in drive_root.glob("*Forward*Risk*Manager*") if p.is_dir())
        if not fuzzy:
            raise FileNotFoundError(
                "Repo not found under /content/drive/MyDrive. "
                "Place the repo in Drive or set REPO_DIR manually in this cell."
            )
        REPO_DIR = fuzzy[0]
        print("Using discovered repo dir:", REPO_DIR)
else:
    REPO_DIR = os.getcwd()
    print("Non-Colab runtime detected. Using current working directory as repo.")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

if Path(".git").exists():
    result = subprocess.run(
        ["git", "status", "--short"],
        text=True,
        capture_output=True,
    )
    if result.returncode == 0:
        output = result.stdout.strip()
        print(output[:4000] if output else "Git repo detected. Working tree clean.")
    else:
        msg = (result.stderr or result.stdout or "").strip()
        print(f"`git status --short` failed with exit code {result.returncode}.")
        if msg:
            print(msg[:4000])

        low = msg.lower()
        if "dubious ownership" in low:
            print('Fix: run `!git config --global --add safe.directory "$PWD"` and retry.')
        elif "not a git repository" in low:
            print("The folder has .git metadata but Git cannot resolve a valid repository here.")
else:
    print("No .git directory found. Running from copied folder is fine.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cwd: /content/drive/MyDrive/Forward-Risk-Manager
M configs/train_long_constituents.toml
 M notebooks/colab_setup.ipynb
 M scripts/benchmark_training.py
 M scripts/dual_score_report.py
 M scripts/ff_sweep.py
 M scripts/hallucination_calibration.py
 M scripts/promote_sweep_best.py
 M scripts/publish_run.py
 M scripts/qc_export_to_tidy.py
 M scripts/sanity_checks.py
 M scripts/scenario_book.py
 M scripts/stress_test_report.py
 M src/frisk/econ_eval.py
 M tests/test_econ_eval.py
?? notebooks/reduced_smoke_test.ipynb
?? tests/test_reporting_scripts.py


## 3) Install Dependencies And Verify Tests


In [10]:
import shlex
import subprocess
import sys
from pathlib import Path

PYTHON = shlex.quote(sys.executable)
RUN_TESTS = True

commands = [
    f"{PYTHON} -m pip install --upgrade pip setuptools wheel",
    f"{PYTHON} -m pip install -r requirements.txt",
]

if sys.version_info < (3, 11):
    commands.append(f"{PYTHON} -m pip install tomli")

commands.extend(
    [
        f"{PYTHON} -m pip install pytest",
        # --no-build-isolation avoids unnecessary build-dep fetches in constrained environments
        f"{PYTHON} -m pip install -e . --no-build-isolation",
    ]
)

if RUN_TESTS:
    commands.append(f"{PYTHON} -m pytest -q")
else:
    print("RUN_TESTS=False -> skipping pytest.")

for cmd in commands:
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=True)

compat_path = Path("tomllib.py")
if sys.version_info < (3, 11):
    compat_path.write_text(
        "# Compatibility shim for Python < 3.11.\n"
        "from tomli import TOMLDecodeError, load, loads\n"
    )
    import tomli as _tomli

    sys.modules["tomllib"] = _tomli
    print("Created tomllib compatibility shim:", compat_path)
elif compat_path.exists():
    compat_path.unlink()
    print("Removed stale tomllib compatibility shim:", compat_path)

print("Dependency + test setup complete.")


+ /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
+ /usr/bin/python3 -m pip install -r requirements.txt
+ /usr/bin/python3 -m pip install pytest
+ /usr/bin/python3 -m pip install -e . --no-build-isolation
+ /usr/bin/python3 -m pytest -q
Dependency + test setup complete.


## 4) Intensive Session Controls


In [11]:
from pathlib import Path

try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

# Base configs
BUILD_CONFIG = "configs/long_constituents.toml"
TRAIN_CONFIG = "configs/train_long_constituents.toml"
SWEEP_SECTION = "sweep"
SWEEP_E2E_SECTION = "sweep_e2e"
DEVICE = "cuda"

# Run layout
RUN_ID = "long_constituents"
RUN_ROOT = f"runs/experiments/{RUN_ID}"
PUBLISHED_ROOT = "reports/published"
REPORT_INDEX_CSV = "reports/index.csv"
RUNTIME_TRAIN_CONFIG = f"{RUN_ROOT}/runtime/train_config_runtime.toml"

# Evaluation semantics for benchmark/sweep
# - self_contrastive + eval_neg_mode='auto' reports retrieval accuracy (eval_sc_acc / eval_acc),
#   which often saturates near 1.0 and is not directly comparable to FF eval_sep.
# - set eval_neg_mode='shuffle' for a comparable FF-style eval across modes.
BENCH_EVAL_NEG_MODE = "auto"   # 'shuffle' | 'self_contrastive' | 'auto'
SWEEP_EVAL_NEG_MODE = "auto"   # 'shuffle' | 'self_contrastive' | 'auto'
SC_EVAL_VIEW_MODE = "noise"
SC_EVAL_NOISE_STD = 0.05
SC_TRAIN_VIEW_MODE = "noise"
SC_TRAIN_NOISE_STD = 0.05
TRAIN_NEG_MODE = "time_flip+noise"
TRAIN_NOISE_STD = 0.05
HARD_NEG_MODE_LAYERWISE = "time_flip+noise"
HARD_NEG_MODE_BACKPROP = "time_flip+noise"
# Keep default extra eval modes lean for faster Colab runs.
# You can add ",block_bootstrap,cross_asset_mix,phase_randomize" when you want fuller diagnostics.
BENCH_EVAL_NEG_MODES = "shuffle,time_flip"
SWEEP_EVAL_NEG_MODES = "time_flip"
EVAL_ECE_BINS = 10

# Canonical artifact paths for this notebook run
BENCHMARK_CSV = f"{RUN_ROOT}/metrics/benchmark.csv"
SWEEP_CSV = f"{RUN_ROOT}/metrics/ff_sweep.csv"
SWEEP_E2E_CSV = f"{RUN_ROOT}/metrics/ff_sweep_e2e.csv"
DUAL_SCORE_CSV = f"{RUN_ROOT}/metrics/dual_score_report.csv"
DUAL_SCORE_TXT = f"{RUN_ROOT}/logs/dual_score_report.txt"
SCENARIO_CSV = f"{RUN_ROOT}/metrics/scenario_book.csv"
SCENARIO_DIAG_CSV = f"{RUN_ROOT}/diagnostics/scenario_constraint_diagnostics.csv"
STRESS_CSV = f"{RUN_ROOT}/metrics/stress_test_report.csv"
STRESS_PLOT = f"{RUN_ROOT}/plots/stress_test_report.png"
HALL_CAL_JSON = f"{RUN_ROOT}/diagnostics/hallucination_calibration.json"
HALL_CAL_BY_TICKER = f"{RUN_ROOT}/diagnostics/hallucination_calibration_by_ticker.csv"
HALL_WINDOW_CSV = f"{RUN_ROOT}/diagnostics/hallucination_window.csv"
HALL_WINDOW_ALL_CSV = f"{RUN_ROOT}/diagnostics/hallucination_window_all.csv"
HALL_PLOT_PNG = f"{RUN_ROOT}/diagnostics/hallucination_plot.png"
HALL_DIAG_PNG = f"{RUN_ROOT}/diagnostics/hallucination_diagnostics.png"
GOODNESS_CSV = f"{RUN_ROOT}/diagnostics/goodness_backtest.csv"
GOODNESS_QUANTILES_CSV = f"{RUN_ROOT}/diagnostics/goodness_quantiles.csv"
GOODNESS_SCATTER_PNG = f"{RUN_ROOT}/diagnostics/goodness_scatter.png"
GOODNESS_EVENTS_CSV = f"{RUN_ROOT}/diagnostics/goodness_events.csv"
GOODNESS_STRATEGY_CSV = f"{RUN_ROOT}/diagnostics/goodness_strategy_metrics.csv"
GOODNESS_TIMELINE_PNG = f"{RUN_ROOT}/diagnostics/goodness_timeline.png"
SWEEP_SUMMARY_TXT = f"{RUN_ROOT}/logs/ff_sweep_summary.txt"
SWEEP_E2E_SUMMARY_TXT = f"{RUN_ROOT}/logs/ff_sweep_e2e_summary.txt"
SWEEP_TRADEOFF_PNG = f"{RUN_ROOT}/plots/ff_sweep_tradeoff.png"
SWEEP_PARETO_PNG = f"{RUN_ROOT}/plots/ff_sweep_pareto.png"
ARTIFACT_BUNDLE = f"{RUN_ROOT}/bundles/colab_intensive_bundle.zip"

# Override train epochs at runtime for a deeper Colab pass
TRAIN_EPOCHS_OVERRIDE = 280

# Pipeline toggles
RUN_MERGE_RAW = False
RUN_BUILD = True
RUN_TRAIN = True
RUN_BENCHMARK = True
RUN_SANITY_CHECKS = True
SANITY_BLOCKING = True
SANITY_EASY_NEG_ACC_MAX = 0.995
SANITY_TIMEFLIP_SEP_MIN = 0.05
SANITY_SC_GAP_MIN = 0.2
SANITY_SC_TIMEFLIP_ENABLED = False
SANITY_SC_TIMEFLIP_SEP_MIN = 0.0
SANITY_SC_TIMEFLIP_AUROC_MIN = 0.5
RUN_SWEEP = True
RUN_SWEEP_SUMMARY = True
RUN_SWEEP_E2E_FOCUSED = True
RUN_DUAL_SCORE_REPORT = True
RUN_PLOT_SWEEP = True
RUN_PROMOTE_SWEEP = True
RUN_RETRAIN_AFTER_PROMOTE = True
RUN_REBENCHMARK_AFTER_PROMOTE = True
RUN_GOODNESS_BACKTEST = True
RUN_SCENARIO_BOOK = True
RUN_STRESS_TEST_REPORT = True
RUN_HALLUCINATION_CALIBRATION = True
RUN_PLOT_HALLUCINATION = True
RUN_HALLUCINATION_DIAGNOSTICS = True
RUN_PUBLISH_ARTIFACTS = True
RUN_EXPORT_ARTIFACTS = True

ALLOW_OPTIONAL_FAILURES = True

# Optional data prep
RAW_ROOT = "data/raw"
MERGED_RAW_DIR = "data/raw_merged"

# Leakage-safe graph build lags (set to 1 for forecasting-safe topology/features)
BUILD_CORR_LAG_DAYS = 1
BUILD_FEATURE_LAG_DAYS = 1
BUILD_MEMBERSHIP_LAG_DAYS = 1

# Sweep promotion
PROMOTE_SWEEP_RANK_BY = "auto"   # auto | primary_eval_metric_robust | rank_value | score | eval_sc_gap | eval_sep | eval_acc | graphs_per_s
PROMOTE_SWEEP_MODE = ""          # e.g. 'ff_e2e' or ''
PROMOTE_SWEEP_APPLY_MODE = True

# Goodness backtest
BACKTEST_TICKER = "AUTO"
BACKTEST_FALLBACK_TICKER = ""
BACKTEST_HORIZONS = "5,21"
BACKTEST_MAX_ABS_LOGRET = 0.35

# Scenario generation (exact-target constraints)
SCENARIO_TICKER = "AUTO"
SCENARIO_FALLBACK_TICKER = ""
SCENARIO_NUM = 150
SCENARIO_TARGET_DROP = -0.10
SCENARIO_CONSTRAINT_WEIGHT = 60.0
SCENARIO_CONSTRAINT_MODE = "exact"       # exact | at_least
SCENARIO_CONSTRAINT_TOLERANCE = 0.01
SCENARIO_TARGET_TOLERANCE = 0.02
SCENARIO_TARGET_HIT_RATE = 0.85
SCENARIO_MAX_ADAPT_STEPS = 18
SCENARIO_NONTARGET_DRIFT_WEIGHT = 4.0
SCENARIO_NONTARGET_DRIFT_TOLERANCE = 0.01
SCENARIO_MAX_NONTARGET_DRIFT = 0.03
SCENARIO_ADAPT_NONTARGET_MULT = 1.4
SCENARIO_ADAPT_MAX_NONTARGET_WEIGHT = 300.0
SCENARIO_ADAPT_NONTARGET_REG_MULT = 1.15

for p in [BUILD_CONFIG, TRAIN_CONFIG]:
    if not Path(p).exists():
        raise FileNotFoundError(p)

with Path(BUILD_CONFIG).open("rb") as f:
    build_cfg_full = tomllib.load(f)
with Path(TRAIN_CONFIG).open("rb") as f:
    cfg = tomllib.load(f)

build_cfg = build_cfg_full.get("build_graphs", {})
required_build_inputs = [
    build_cfg.get("prices", ""),
    build_cfg.get("constituents", ""),
]
feature_mode = str(build_cfg.get("feature_mode", ""))
if feature_mode.endswith("_fund"):
    required_build_inputs.append(build_cfg.get("fundamentals", ""))
missing_build_inputs = [p for p in required_build_inputs if p and not Path(p).exists()]

graphs_path = Path(str(cfg.get("train", {}).get("graphs", "")))

print("build config:", BUILD_CONFIG)
print("train config:", TRAIN_CONFIG)
print("run root:", RUN_ROOT)
print("benchmark.eval_neg_mode (runtime):", BENCH_EVAL_NEG_MODE)
print("sweep.eval_neg_mode (runtime):", SWEEP_EVAL_NEG_MODE)
print("self_contrastive_eval_view (runtime):", SC_EVAL_VIEW_MODE, "noise_std=", SC_EVAL_NOISE_STD)
print("self_contrastive_train_view (runtime):", SC_TRAIN_VIEW_MODE, "noise_std=", SC_TRAIN_NOISE_STD)
print("train.neg_mode (runtime):", TRAIN_NEG_MODE, "| noise_std=", TRAIN_NOISE_STD)
print("extra eval neg modes (runtime benchmark):", BENCH_EVAL_NEG_MODES)
print(
    "sanity thresholds:",
    {
        "easy_neg_acc_max": SANITY_EASY_NEG_ACC_MAX,
        "timeflip_sep_min": SANITY_TIMEFLIP_SEP_MIN,
        "sc_gap_min": SANITY_SC_GAP_MIN,
        "sc_timeflip_enabled": SANITY_SC_TIMEFLIP_ENABLED,
        "sc_timeflip_sep_min": SANITY_SC_TIMEFLIP_SEP_MIN,
        "sc_timeflip_auroc_min": SANITY_SC_TIMEFLIP_AUROC_MIN,
        "blocking": SANITY_BLOCKING,
    },
)
print("extra eval neg modes (runtime sweep):", SWEEP_EVAL_NEG_MODES)
print("ece bins (runtime):", EVAL_ECE_BINS)
print("backtest ticker request:", BACKTEST_TICKER)
print("scenario ticker request:", SCENARIO_TICKER)
print("train.neg_mode:", cfg.get("train", {}).get("neg_mode", "n/a"))
print("train.goodness_target:", cfg.get("train", {}).get("goodness_target", "n/a"))
print("train.self_contrastive_temp:", cfg.get("train", {}).get("self_contrastive_temp", "n/a"))
print("train.distance_forward_weight:", cfg.get("train", {}).get("distance_forward_weight", "n/a"))
print("graphs path:", graphs_path)

if missing_build_inputs:
    print("Missing build inputs:")
    for p in missing_build_inputs:
        print(" -", p)
    if RUN_BUILD and not RUN_MERGE_RAW:
        raise FileNotFoundError(
            "Build inputs are missing. Set RUN_MERGE_RAW=True or fix paths in BUILD_CONFIG before running."
        )

if RUN_TRAIN and not RUN_BUILD and not graphs_path.exists():
    raise FileNotFoundError(
        f"Training requested but graphs file is missing: {graphs_path}. "
        "Enable RUN_BUILD or provide a valid prebuilt graphs path in TRAIN_CONFIG."
    )


build config: configs/long_constituents.toml
train config: configs/train_long_constituents.toml
run root: runs/experiments/long_constituents
benchmark.eval_neg_mode (runtime): auto
sweep.eval_neg_mode (runtime): auto
self_contrastive_eval_view (runtime): noise noise_std= 0.05
self_contrastive_train_view (runtime): noise noise_std= 0.05
train.neg_mode (runtime): time_flip+noise | noise_std= 0.05
extra eval neg modes (runtime benchmark): shuffle,time_flip
sanity thresholds: {'easy_neg_acc_max': 0.995, 'timeflip_sep_min': 0.05, 'sc_timeflip_sep_min': 0.0, 'sc_timeflip_auroc_min': 0.5, 'blocking': True}
extra eval neg modes (runtime sweep): time_flip
ece bins (runtime): 10
backtest ticker request: AUTO
scenario ticker request: AUTO
train.neg_mode: time_flip+noise
train.goodness_target: 2.697421237500001
train.self_contrastive_temp: 0.15
train.distance_forward_weight: 0.02
graphs path: data/processed_long/graphs_constituents.pt


## 5) Helpers


In [12]:
import os
import re
import csv
import json as _json
import shlex
import subprocess
import sys
import time
import torch
from collections import Counter
from pathlib import Path

try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

PYTHON = shlex.quote(sys.executable)
AUTO_TICKER_VALUES = {"", "AUTO", "AUTO_DETECT", "AUTO-DETECT"}


def q(s: str) -> str:
    return shlex.quote(str(s))


def run(cmd: str, allow_fail: bool = False) -> bool:
    print()
    print("=" * 100)
    print(cmd)
    print("=" * 100)
    t0 = time.time()

    env = os.environ.copy()
    env.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

    p = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()

    dt = time.time() - t0
    if rc != 0:
        msg = f"Command failed ({rc}): {cmd}"
        if allow_fail:
            print("WARNING:", msg)
            return False
        raise RuntimeError(msg)

    print(f"Completed in {dt / 60:.2f} min")
    return True


def _toml_value_literal(value):
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, int) and not isinstance(value, bool):
        return str(value)
    if isinstance(value, float):
        return repr(float(value))
    return _json.dumps(str(value))


def write_runtime_train_config(base_config_path: str, runtime_config_path: str, section_overrides: dict) -> str:
    src = Path(base_config_path)
    dst = Path(runtime_config_path)
    lines = src.read_text().splitlines()

    section_re = re.compile(r"^\s*\[([^\]]+)\]\s*$")
    key_res = {
        sec: {k: re.compile(rf"^\s*{re.escape(k)}\s*=") for k in kv}
        for sec, kv in section_overrides.items()
    }
    replaced = {(sec, key): False for sec, kv in section_overrides.items() for key in kv}
    seen_sections = set()

    out = []
    current_section = None

    def flush_missing(section_name: str | None):
        if section_name not in section_overrides:
            return
        for key, value in section_overrides[section_name].items():
            if not replaced[(section_name, key)]:
                out.append(f"{key} = {_toml_value_literal(value)}")
                replaced[(section_name, key)] = True

    for line in lines:
        m = section_re.match(line)
        if m:
            flush_missing(current_section)
            current_section = m.group(1).strip()
            seen_sections.add(current_section)
            out.append(line)
            continue

        updated = False
        if current_section in section_overrides:
            for key, pattern in key_res[current_section].items():
                if pattern.match(line):
                    out.append(f"{key} = {_toml_value_literal(section_overrides[current_section][key])}")
                    replaced[(current_section, key)] = True
                    updated = True
                    break
        if not updated:
            out.append(line)

    flush_missing(current_section)

    for section_name, kv in section_overrides.items():
        if section_name in seen_sections:
            continue
        out.append("")
        out.append(f"[{section_name}]")
        for key, value in kv.items():
            out.append(f"{key} = {_toml_value_literal(value)}")
            replaced[(section_name, key)] = True

    dst.parent.mkdir(parents=True, exist_ok=True)
    dst.write_text("\n".join(out) + "\n")
    return str(dst)


def normalize_ticker(value: str) -> str:
    return str(value or "").strip().upper()


def choose_backtest_ticker(requested: str, fallback: str) -> str:
    req = normalize_ticker(requested)
    fb = normalize_ticker(fallback)
    if req and req not in AUTO_TICKER_VALUES:
        return req
    if fb and fb not in AUTO_TICKER_VALUES:
        return fb
    return "AUTO"


def choose_scenario_ticker(train_config_path: str, requested: str, fallback: str) -> str:
    req = normalize_ticker(requested)
    fb = normalize_ticker(fallback)

    if req in AUTO_TICKER_VALUES:
        return "AUTO"

    with Path(train_config_path).open("rb") as f:
        cfg = tomllib.load(f)
    graphs_path = Path(cfg.get("train", {}).get("graphs", ""))
    if not graphs_path.exists():
        return req or fb or "AUTO"

    try:
        payload = torch.load(graphs_path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(graphs_path, map_location="cpu")

    tickers_list = payload.get("tickers", []) if isinstance(payload, dict) else []
    counts: Counter[str] = Counter()
    for tickers in tickers_list:
        for t in set(tickers or []):
            tt = normalize_ticker(t)
            if tt:
                counts[tt] += 1

    if not counts:
        return req or fb or "AUTO"

    if req in counts:
        return req

    if fb and fb not in AUTO_TICKER_VALUES and fb in counts:
        print(f"[scenario_book] {req or requested} not found. Using fallback {fb}.")
        return fb

    best_ticker, best_windows = min(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    print(
        f"[scenario_book] {req or requested} not found. "
        f"Using {best_ticker} (present in {best_windows} graph windows)."
    )
    return best_ticker


def resolve_scenario_ticker_for_reports(scenario_csv_path: str, diag_csv_path: str, requested: str) -> str:
    req = normalize_ticker(requested)

    scenario_target = ""
    scenario_first_ticker = ""
    p = Path(scenario_csv_path)
    if p.exists():
        try:
            with p.open(newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    tt = normalize_ticker(row.get("target_ticker", ""))
                    if tt and tt not in AUTO_TICKER_VALUES:
                        if scenario_target and scenario_target != tt:
                            raise RuntimeError(
                                f"[scenario_book] inconsistent target_ticker values in {p}: "
                                f"{scenario_target} vs {tt}"
                            )
                        scenario_target = tt
                    if not scenario_first_ticker:
                        tk = normalize_ticker(row.get("ticker", ""))
                        if tk and tk not in AUTO_TICKER_VALUES:
                            scenario_first_ticker = tk
        except Exception as exc:
            raise RuntimeError(f"[scenario_book] could not infer report ticker from {p}: {exc}") from exc

    diag_target = ""
    d = Path(diag_csv_path)
    if d.exists():
        try:
            with d.open(newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    tt = normalize_ticker(row.get("ticker", ""))
                    if tt and tt not in AUTO_TICKER_VALUES:
                        if diag_target and diag_target != tt:
                            raise RuntimeError(
                                f"[scenario_book] inconsistent diagnostic tickers in {d}: "
                                f"{diag_target} vs {tt}"
                            )
                        diag_target = tt
        except Exception as exc:
            raise RuntimeError(f"[scenario_book] could not read diagnostics ticker from {d}: {exc}") from exc

    if scenario_target and diag_target and scenario_target != diag_target:
        raise RuntimeError(
            "[scenario_book] target ticker mismatch between scenario and diagnostics: "
            f"scenario={scenario_target}, diag={diag_target}"
        )

    if req and req not in AUTO_TICKER_VALUES:
        if scenario_target and req != scenario_target:
            raise RuntimeError(
                "[scenario_book] requested scenario ticker does not match scenario output: "
                f"requested={req}, scenario={scenario_target}"
            )
        if diag_target and req != diag_target:
            raise RuntimeError(
                "[scenario_book] requested scenario ticker does not match diagnostics: "
                f"requested={req}, diag={diag_target}"
            )
        return req

    effective = scenario_target or diag_target or scenario_first_ticker
    if effective:
        print(f"[scenario_book] effective ticker from artifacts: {effective}")
    return effective


def run_step(
    enabled: bool,
    label: str,
    cmd: str,
    *,
    allow_fail: bool = False,
    requires: list[str] | None = None,
    script_path: str = "",
) -> bool:
    if not enabled:
        return False

    missing = []
    for rp in requires or []:
        if rp and not Path(rp).exists():
            missing.append(rp)
    if script_path and not Path(script_path).exists():
        missing.append(script_path)

    if missing:
        print(f"Skipping {label}: missing prerequisites -> {', '.join(str(x) for x in missing)}")
        return False

    return run(cmd, allow_fail=allow_fail)


## 6) Run Full Intensive Pipeline


In [ ]:
runtime_overrides = {
    "train": {
        "neg_mode": TRAIN_NEG_MODE,
        "noise_std": float(TRAIN_NOISE_STD),
    },
    "benchmark": {
        "epochs": 30,
        "eval_neg_mode": BENCH_EVAL_NEG_MODE,
        "eval_neg_modes": BENCH_EVAL_NEG_MODES,
        "self_contrastive_eval_view_mode": SC_EVAL_VIEW_MODE,
        "self_contrastive_eval_noise_std": float(SC_EVAL_NOISE_STD),
        "ece_bins": int(EVAL_ECE_BINS),
    },
    "benchmark.mode_overrides.ff_layerwise": {
        "neg_mode": HARD_NEG_MODE_LAYERWISE,
        "eval_neg_mode": HARD_NEG_MODE_LAYERWISE,
    },
    "benchmark.mode_overrides.ff_e2e": {
        "self_contrastive_view_mode": SC_TRAIN_VIEW_MODE,
        "self_contrastive_view_noise_std": float(SC_TRAIN_NOISE_STD),
        "self_contrastive_eval_view_mode": SC_EVAL_VIEW_MODE,
        "self_contrastive_eval_noise_std": float(SC_EVAL_NOISE_STD),
    },
    "benchmark.mode_overrides.backprop": {
        "neg_mode": HARD_NEG_MODE_BACKPROP,
        "eval_neg_mode": HARD_NEG_MODE_BACKPROP,
    },
    "sweep": {
        "eval_neg_mode": SWEEP_EVAL_NEG_MODE,
        "eval_neg_modes": SWEEP_EVAL_NEG_MODES,
        "self_contrastive_eval_view_mode": SC_EVAL_VIEW_MODE,
        "self_contrastive_eval_noise_std": float(SC_EVAL_NOISE_STD),
        "ece_bins": int(EVAL_ECE_BINS),
    },
    "sweep.mode_overrides.ff_layerwise": {
        "neg_mode": HARD_NEG_MODE_LAYERWISE,
        "eval_neg_mode": HARD_NEG_MODE_LAYERWISE,
    },
    "sweep.mode_overrides.ff_e2e": {
        "self_contrastive_view_mode": SC_TRAIN_VIEW_MODE,
        "self_contrastive_view_noise_std": float(SC_TRAIN_NOISE_STD),
        "self_contrastive_eval_view_mode": SC_EVAL_VIEW_MODE,
        "self_contrastive_eval_noise_std": float(SC_EVAL_NOISE_STD),
    },
    "sweep_e2e": {
        "eval_neg_mode": "self_contrastive",
        "eval_neg_modes": SWEEP_EVAL_NEG_MODES,
        "self_contrastive_eval_view_mode": SC_EVAL_VIEW_MODE,
        "self_contrastive_eval_noise_std": float(SC_EVAL_NOISE_STD),
        "ece_bins": int(EVAL_ECE_BINS),
    },
    "sweep_e2e.mode_overrides.ff_e2e": {
        "self_contrastive_view_mode": SC_TRAIN_VIEW_MODE,
        "self_contrastive_view_noise_std": float(SC_TRAIN_NOISE_STD),
        "self_contrastive_eval_view_mode": SC_EVAL_VIEW_MODE,
        "self_contrastive_eval_noise_std": float(SC_EVAL_NOISE_STD),
    },
}
ACTIVE_TRAIN_CONFIG = write_runtime_train_config(TRAIN_CONFIG, RUNTIME_TRAIN_CONFIG, runtime_overrides)
print("active train config:", ACTIVE_TRAIN_CONFIG)

with Path(ACTIVE_TRAIN_CONFIG).open("rb") as f:
    runtime_cfg = tomllib.load(f)
runtime_graphs = str(runtime_cfg.get("train", {}).get("graphs", ""))
SANITY_GATE_PASSED = True

if RUN_MERGE_RAW:
    run_step(
        True,
        "merge_raw_years",
        f"{PYTHON} scripts/merge_raw_years.py --raw-root {q(RAW_ROOT)} --out-dir {q(MERGED_RAW_DIR)}",
        script_path="scripts/merge_raw_years.py",
    )

if RUN_BUILD:
    build_cmd = (
        f"{PYTHON} scripts/build_graphs.py --config {q(BUILD_CONFIG)} "
        f"--corr-lag-days {int(BUILD_CORR_LAG_DAYS)} "
        f"--feature-lag-days {int(BUILD_FEATURE_LAG_DAYS)} "
        f"--membership-lag-days {int(BUILD_MEMBERSHIP_LAG_DAYS)}"
    )
    run_step(True, "build_graphs", build_cmd, script_path="scripts/build_graphs.py", requires=[BUILD_CONFIG])

train_cmd = f"{PYTHON} scripts/train_ff_gnn.py --config {q(ACTIVE_TRAIN_CONFIG)} --device {q(DEVICE)}"
if TRAIN_EPOCHS_OVERRIDE and int(TRAIN_EPOCHS_OVERRIDE) > 0:
    train_cmd += f" --epochs {int(TRAIN_EPOCHS_OVERRIDE)}"

if RUN_TRAIN:
    train_requires = [ACTIVE_TRAIN_CONFIG]
    if runtime_graphs and not RUN_BUILD:
        train_requires.append(runtime_graphs)
    run_step(True, "train_ff_gnn", train_cmd, script_path="scripts/train_ff_gnn.py", requires=train_requires)

if RUN_BENCHMARK:
    run_step(
        True,
        "benchmark_training",
        f"{PYTHON} scripts/benchmark_training.py --config {q(ACTIVE_TRAIN_CONFIG)}",
        allow_fail=ALLOW_OPTIONAL_FAILURES,
        script_path="scripts/benchmark_training.py",
        requires=[ACTIVE_TRAIN_CONFIG],
    )


sanity_parts = [
    f"{PYTHON} scripts/sanity_checks.py",
    f"--benchmark-csv {q(BENCHMARK_CSV)}",
    f"--easy-neg-acc-max {float(SANITY_EASY_NEG_ACC_MAX)}",
    f"--timeflip-sep-min {float(SANITY_TIMEFLIP_SEP_MIN)}",
    f"--sc-gap-min {float(SANITY_SC_GAP_MIN)}",
]
if SANITY_SC_TIMEFLIP_ENABLED:
    sanity_parts.append(f"--sc-timeflip-sep-min {float(SANITY_SC_TIMEFLIP_SEP_MIN)}")
    sanity_parts.append(f"--sc-timeflip-auroc-min {float(SANITY_SC_TIMEFLIP_AUROC_MIN)}")
else:
    sanity_parts.append("--skip-sc-timeflip-checks")
sanity_cmd = " ".join(sanity_parts)

if RUN_SANITY_CHECKS:
    sanity_pre_passed = run_step(
        True,
        "sanity_checks",
        sanity_cmd,
        allow_fail=True,
        script_path="scripts/sanity_checks.py",
        requires=[BENCHMARK_CSV],
    )
    if not sanity_pre_passed:
        print("WARNING: sanity_checks failed before sweep; continuing. Blocking sanity gate is enforced post-promote/rebenchmark.")

if RUN_SWEEP:
    run_step(
        True,
        "ff_sweep",
        f"{PYTHON} scripts/ff_sweep.py --config {q(ACTIVE_TRAIN_CONFIG)} --section {q(SWEEP_SECTION)}",
        script_path="scripts/ff_sweep.py",
        requires=[ACTIVE_TRAIN_CONFIG],
    )

if RUN_SWEEP_E2E_FOCUSED:
    run_step(
        True,
        "ff_sweep_e2e",
        f"{PYTHON} scripts/ff_sweep.py --config {q(ACTIVE_TRAIN_CONFIG)} --section {q(SWEEP_E2E_SECTION)}",
        allow_fail=ALLOW_OPTIONAL_FAILURES,
        script_path="scripts/ff_sweep.py",
        requires=[ACTIVE_TRAIN_CONFIG],
    )

if RUN_SWEEP_SUMMARY:
    run_step(
        True,
        "ff_sweep_summary",
        f"{PYTHON} scripts/ff_sweep_summary.py --csv {q(SWEEP_CSV)} --out {q(SWEEP_SUMMARY_TXT)}",
        script_path="scripts/ff_sweep_summary.py",
        requires=[SWEEP_CSV],
    )
    if RUN_SWEEP_E2E_FOCUSED:
        run_step(
            True,
            "ff_sweep_e2e_summary",
            f"{PYTHON} scripts/ff_sweep_summary.py --csv {q(SWEEP_E2E_CSV)} --out {q(SWEEP_E2E_SUMMARY_TXT)}",
            allow_fail=ALLOW_OPTIONAL_FAILURES,
            script_path="scripts/ff_sweep_summary.py",
            requires=[SWEEP_E2E_CSV],
        )

if RUN_PLOT_SWEEP:
    run_step(
        True,
        "plot_ff_sweep",
        f"{PYTHON} scripts/plot_ff_sweep.py --csv {q(SWEEP_CSV)} "
        f"--out {q(SWEEP_TRADEOFF_PNG)} --pareto-out {q(SWEEP_PARETO_PNG)}",
        script_path="scripts/plot_ff_sweep.py",
        requires=[SWEEP_CSV],
    )

promoted = False
if RUN_PROMOTE_SWEEP:
    mode_arg = f" --mode {q(PROMOTE_SWEEP_MODE)}" if PROMOTE_SWEEP_MODE else ""
    apply_mode_arg = "--apply-mode" if PROMOTE_SWEEP_APPLY_MODE else "--no-apply-mode"
    promoted = run_step(
        True,
        "promote_sweep_best",
        f"{PYTHON} scripts/promote_sweep_best.py --config {q(ACTIVE_TRAIN_CONFIG)} "
        f"--csv {q(SWEEP_CSV)} --rank-by {q(PROMOTE_SWEEP_RANK_BY)} "
        f"{apply_mode_arg}{mode_arg} --apply",
        script_path="scripts/promote_sweep_best.py",
        requires=[ACTIVE_TRAIN_CONFIG, SWEEP_CSV],
    )

if promoted and RUN_RETRAIN_AFTER_PROMOTE:
    run_step(True, "retrain_after_promote", train_cmd, script_path="scripts/train_ff_gnn.py", requires=[ACTIVE_TRAIN_CONFIG])
    if RUN_REBENCHMARK_AFTER_PROMOTE:
        run_step(
            True,
            "rebenchmark_after_promote",
            f"{PYTHON} scripts/benchmark_training.py --config {q(ACTIVE_TRAIN_CONFIG)}",
            allow_fail=ALLOW_OPTIONAL_FAILURES,
            script_path="scripts/benchmark_training.py",
            requires=[ACTIVE_TRAIN_CONFIG],
        )
        if RUN_SANITY_CHECKS:
            SANITY_GATE_PASSED = run_step(
                True,
                "sanity_checks_post_promote",
                sanity_cmd,
                allow_fail=(not SANITY_BLOCKING),
                script_path="scripts/sanity_checks.py",
                requires=[BENCHMARK_CSV],
            )

if RUN_DUAL_SCORE_REPORT:
    run_step(
        True,
        "dual_score_report",
        f"{PYTHON} scripts/dual_score_report.py --benchmark {q(BENCHMARK_CSV)} "
        f"--sweep {q(SWEEP_CSV)} --sweep-e2e {q(SWEEP_E2E_CSV)} "
        f"--out {q(DUAL_SCORE_TXT)} --out-csv {q(DUAL_SCORE_CSV)}",
        allow_fail=ALLOW_OPTIONAL_FAILURES,
        script_path="scripts/dual_score_report.py",
        requires=[BENCHMARK_CSV, SWEEP_CSV],
    )

backtest_ticker = choose_backtest_ticker(BACKTEST_TICKER, BACKTEST_FALLBACK_TICKER)
print("goodness_backtest ticker request:", backtest_ticker)

if RUN_GOODNESS_BACKTEST:
    run_step(
        True,
        "goodness_backtest",
        f"{PYTHON} scripts/goodness_backtest.py --config {q(ACTIVE_TRAIN_CONFIG)} "
        f"--ticker {q(backtest_ticker)} --horizons {q(BACKTEST_HORIZONS)} "
        f"--max-abs-logret {float(BACKTEST_MAX_ABS_LOGRET)} "
        f"--out-csv {q(GOODNESS_CSV)} --out-quantiles {q(GOODNESS_QUANTILES_CSV)} "
        f"--out-plot {q(GOODNESS_SCATTER_PNG)} "
        f"--out-events {q(GOODNESS_EVENTS_CSV)} --out-strategy {q(GOODNESS_STRATEGY_CSV)} --out-timeline {q(GOODNESS_TIMELINE_PNG)}",
        allow_fail=ALLOW_OPTIONAL_FAILURES,
        script_path="scripts/goodness_backtest.py",
        requires=[ACTIVE_TRAIN_CONFIG],
    )

scenario_ticker_request = choose_scenario_ticker(ACTIVE_TRAIN_CONFIG, SCENARIO_TICKER, SCENARIO_FALLBACK_TICKER)
print("scenario target ticker request:", scenario_ticker_request)

if RUN_SCENARIO_BOOK:
    run_step(
        True,
        "scenario_book",
        f"{PYTHON} scripts/scenario_book.py --config {q(ACTIVE_TRAIN_CONFIG)} "
        f"--num-scenarios {int(SCENARIO_NUM)} "
        f"--target-ticker {q(scenario_ticker_request)} "
        f"--target-drop {float(SCENARIO_TARGET_DROP)} "
        f"--constraint-weight {float(SCENARIO_CONSTRAINT_WEIGHT)} "
        f"--constraint-mode {q(SCENARIO_CONSTRAINT_MODE)} "
        f"--constraint-tolerance {float(SCENARIO_CONSTRAINT_TOLERANCE)} "
        f"--target-tolerance {float(SCENARIO_TARGET_TOLERANCE)} "
        f"--target-hit-rate {float(SCENARIO_TARGET_HIT_RATE)} "
        f"--max-adapt-steps {int(SCENARIO_MAX_ADAPT_STEPS)} "
        f"--nontarget-drift-weight {float(SCENARIO_NONTARGET_DRIFT_WEIGHT)} "
        f"--nontarget-drift-tolerance {float(SCENARIO_NONTARGET_DRIFT_TOLERANCE)} "
        f"--max-nontarget-drift {float(SCENARIO_MAX_NONTARGET_DRIFT)} "
        f"--adapt-nontarget-mult {float(SCENARIO_ADAPT_NONTARGET_MULT)} "
        f"--adapt-max-nontarget-weight {float(SCENARIO_ADAPT_MAX_NONTARGET_WEIGHT)} "
        f"--adapt-nontarget-reg-mult {float(SCENARIO_ADAPT_NONTARGET_REG_MULT)} "
        f"--adaptive "
        f"--diag-out {q(SCENARIO_DIAG_CSV)} --out {q(SCENARIO_CSV)}",
        allow_fail=ALLOW_OPTIONAL_FAILURES,
        script_path="scripts/scenario_book.py",
        requires=[ACTIVE_TRAIN_CONFIG],
    )

scenario_ticker_effective = resolve_scenario_ticker_for_reports(SCENARIO_CSV, SCENARIO_DIAG_CSV, scenario_ticker_request)
if not scenario_ticker_effective and scenario_ticker_request not in AUTO_TICKER_VALUES:
    scenario_ticker_effective = scenario_ticker_request
print("scenario target ticker effective for reports:", scenario_ticker_effective or "<all>")
scenario_target_arg = f" --target-ticker {q(scenario_ticker_effective)}" if scenario_ticker_effective else ""

if RUN_STRESS_TEST_REPORT:
    run_step(
        True,
        "stress_test_report",
        f"{PYTHON} scripts/stress_test_report.py --csv {q(SCENARIO_CSV)}"
        f"{scenario_target_arg} --out-csv {q(STRESS_CSV)} --out-plot {q(STRESS_PLOT)}",
        allow_fail=ALLOW_OPTIONAL_FAILURES,
        script_path="scripts/stress_test_report.py",
        requires=[SCENARIO_CSV],
    )

if RUN_HALLUCINATION_CALIBRATION:
    run_step(
        True,
        "hallucination_calibration",
        f"{PYTHON} scripts/hallucination_calibration.py --csv {q(SCENARIO_CSV)}"
        f"{scenario_target_arg} --out {q(HALL_CAL_JSON)} --out-by-ticker {q(HALL_CAL_BY_TICKER)}",
        allow_fail=ALLOW_OPTIONAL_FAILURES,
        script_path="scripts/hallucination_calibration.py",
        requires=[SCENARIO_CSV],
    )

if RUN_PLOT_HALLUCINATION:
    run_step(
        True,
        "plot_hallucination",
        f"{PYTHON} scripts/plot_hallucination.py --config {q(ACTIVE_TRAIN_CONFIG)} "
        f"--out {q(HALL_PLOT_PNG)} --save-csv {q(HALL_WINDOW_CSV)} "
        f"--save-csv-all {q(HALL_WINDOW_ALL_CSV)}",
        allow_fail=ALLOW_OPTIONAL_FAILURES,
        script_path="scripts/plot_hallucination.py",
        requires=[ACTIVE_TRAIN_CONFIG],
    )

if RUN_HALLUCINATION_DIAGNOSTICS:
    run_step(
        True,
        "plot_hallucination_diagnostics",
        f"{PYTHON} scripts/plot_hallucination_diagnostics.py "
        f"--csv {q(HALL_WINDOW_ALL_CSV)} --out {q(HALL_DIAG_PNG)}",
        allow_fail=ALLOW_OPTIONAL_FAILURES,
        script_path="scripts/plot_hallucination_diagnostics.py",
        requires=[HALL_WINDOW_ALL_CSV],
    )

if RUN_PUBLISH_ARTIFACTS:
    if RUN_SANITY_CHECKS and SANITY_BLOCKING and not SANITY_GATE_PASSED:
        raise RuntimeError("publish_run blocked: sanity checks failed")
    run_step(
        True,
        "publish_run",
        f"{PYTHON} scripts/publish_run.py --run-id {q(RUN_ID)}",
        allow_fail=False,
        script_path="scripts/publish_run.py",
        requires=[RUN_ROOT],
    )

if RUN_EXPORT_ARTIFACTS:
    bundle_path = Path(ARTIFACT_BUNDLE)
    bundle_path.parent.mkdir(parents=True, exist_ok=True)
    zip_inputs = [pp for pp in (RUN_ROOT, PUBLISHED_ROOT, REPORT_INDEX_CSV) if Path(pp).exists()]
    if not zip_inputs:
        print("Skipping export: no bundle inputs exist yet.")
    else:
        run(
            f"zip -r {q(str(bundle_path))} {' '.join(q(pp) for pp in zip_inputs)} -x 'runs/cache/*'",
            allow_fail=True,
        )

print("Intensive pipeline finished.")


active train config: runs/experiments/long_constituents/runtime/train_config_runtime.toml

/usr/bin/python3 scripts/build_graphs.py --config configs/long_constituents.toml --corr-lag-days 1 --feature-lag-days 1 --membership-lag-days 1
Membership ffill: source_dates=2659 filled_dates=1486 gap_dropped=134 max_gap_days=63

Building graphs: 100%|██████████| 3665/3665 [04:06<00:00, 14.88win/s]
Wrote data/processed_long/graphs_constituents.pt with 3530 graphs
Date range: 2011-06-29 -> 2026-02-06 | windows: 3665 | built: 3530 | skipped_lag=1, skipped: members=134, cols=0, min_nodes=0, no_edges=0
Completed in 5.00 min

/usr/bin/python3 scripts/train_ff_gnn.py --config runs/experiments/long_constituents/runtime/train_config_runtime.toml --device cuda --epochs 280
ff_blockwise requires ff_layerwise; disabling ff_blockwise.
device request: cuda
device: cuda
torch: 2.9.0+cu128
cuda_available: True
cuda_version: 12.8
mps_built: False
mps_available: False
cuda_device_name: Tesla T4
neg_mode: time_fl

## 7) Inspect Key Outputs


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def add_primary_eval_metric(df: pd.DataFrame) -> pd.DataFrame:
    if 'eval_objective' not in df.columns:
        return df
    out = df.copy()
    objective = out['eval_objective'].astype(str).str.lower()
    sep = pd.to_numeric(out['eval_sep'], errors='coerce') if 'eval_sep' in out.columns else np.nan
    sc_gap = pd.to_numeric(out['eval_sc_gap'], errors='coerce') if 'eval_sc_gap' in out.columns else np.nan
    auroc = pd.to_numeric(out['eval_auroc'], errors='coerce') if 'eval_auroc' in out.columns else np.nan
    primary = np.where(objective.eq('self_contrastive'), sc_gap, sep)
    primary = np.where(pd.isna(primary), auroc, primary)
    out['primary_eval_metric'] = primary
    return out


csvs = [
    f'{RUN_ROOT}/metrics/ff_train.csv',
    BENCHMARK_CSV,
    SWEEP_CSV,
    SWEEP_E2E_CSV,
    DUAL_SCORE_CSV,
    SCENARIO_DIAG_CSV,
    STRESS_CSV,
    HALL_CAL_BY_TICKER,
    GOODNESS_CSV,
    GOODNESS_EVENTS_CSV,
    GOODNESS_STRATEGY_CSV,
]

for path in csvs:
    p = Path(path)
    print()
    print('=' * 100)
    print(p)
    if not p.exists():
        print('missing')
        continue
    df = pd.read_csv(p)
    print('shape:', df.shape)

    if p.name in {'benchmark.csv', 'ff_sweep.csv', 'ff_sweep_e2e.csv'}:
        df = add_primary_eval_metric(df)
        cols = [
            c
            for c in [
                'mode',
                'eval_objective',
                'eval_neg_mode_effective',
                'eval_acc',
                'eval_sc_acc',
                'eval_sep',
                'eval_sc_gap',
                'eval_auroc',
                'eval_auprc',
                'eval_brier',
                'eval_ece',
                'eval_neg_modes_reported',
                'primary_eval_metric',
                'avg_epoch_s',
                'graphs_per_s',
            ]
            if c in df.columns
        ]
        extra_neg_cols = sorted(
            c for c in df.columns
            if c.startswith('eval_') and any(k in c for k in ['time_flip', 'block_bootstrap', 'cross_asset_mix', 'phase_randomize'])
        )
        cols = cols + [c for c in extra_neg_cols if c not in cols]
        with pd.option_context('display.max_columns', 80):
            print(df[cols].head(12))
        if (df.get('eval_objective', pd.Series(dtype=str)).astype(str).str.lower() == 'self_contrastive').any():
            print(
                'note: self_contrastive rows use retrieval accuracy (eval_sc_acc / eval_acc), '
                'which can be close to 1.0. Compare primary_eval_metric '
                '(eval_sc_gap for self_contrastive, eval_sep otherwise).'
            )
    else:
        with pd.option_context('display.max_columns', 80):
            print(df.head(3))

print()
print('Run artifacts preview (first 200 files):')
for p in sorted(Path(RUN_ROOT).rglob('*'))[:200]:
    if p.is_file():
        print(p)

print()
print('Published artifacts preview (first 200 files):')
for p in sorted(Path(PUBLISHED_ROOT).rglob('*'))[:200]:
    if p.is_file():
        print(p)


## 8) Preview Plots


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

images = [
    f'{RUN_ROOT}/plots/ff_train.png',
    f'{RUN_ROOT}/plots/benchmark.png',
    f'{RUN_ROOT}/plots/benchmark_speed_sep.png',
    SWEEP_TRADEOFF_PNG,
    SWEEP_PARETO_PNG,
    GOODNESS_SCATTER_PNG,
    GOODNESS_TIMELINE_PNG,
    STRESS_PLOT,
    HALL_DIAG_PNG,
]

for img_path in images:
    p = Path(img_path)
    if not p.exists():
        continue
    img = Image.open(p)
    plt.figure(figsize=(11, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title(str(p))
    plt.show()



## 9) Download Artifact Bundle


In [ ]:
from pathlib import Path

bundle = Path(ARTIFACT_BUNDLE)
if bundle.exists():
    print('bundle:', bundle, 'size_mb=', round(bundle.stat().st_size / (1024 * 1024), 2))
    try:
        from google.colab import files
        files.download(str(bundle))
    except Exception as exc:
        print('Auto-download unavailable in this runtime:', exc)
else:
    print('Artifact bundle not found:', bundle)


## Notes

- This notebook is configured for a heavy Colab T4 session; long cells can run for hours.
- Ticker controls now default to `AUTO` to avoid hard dependence on `MDY`.
- On Python 3.10 runtimes, setup writes a temporary `tomllib.py` shim for script compatibility.
- `RUN_PROMOTE_SWEEP=True` mutates the runtime config copy (`RUNTIME_TRAIN_CONFIG`), not the base train config file.
- Optional stages are skipped automatically when required inputs are missing.
- If CUDA OOM occurs, reduce batch size in `TRAIN_CONFIG` and rerun from training onward.
- Keep runtime connected during intensive runs to avoid kernel reset.
- Sweep promotion in `auto` mode uses objective-aware rank (`rank_value`) when available.
